# 06 · Query-based: RT-DETR

**Model.** Ultralytics RT-DETR-L (HGNetv2 backbone, hybrid encoder, 6-layer decoder, 300 queries),
COCO-pretrained. **Budget.** Same as YOLO11s in notebook 05: 2 GPU-hours (`time: 1.0` on 2× T4),
same frames and augmentation pipeline; AdamW lr 1e-4.

| run | change | motivation |
|---|---|---|
| `rtdetr_base` | defaults: 300 queries, matching cost class 2 / L1 5 / GIoU 2 | reference |
| `rtdetr_q500` | 500 queries | up to 180 objects per frame; 500 detections are scored |
| `rtdetr_matcher` | matching cost GIoU 5 instead of 2 | L1 on normalised coordinates is tiny for small boxes, so GIoU separates queries better |
| `rtdetr_bblr` | backbone lr × 0.1 | keep the pretrained features while the transformer adapts |

## Bipartite matching instead of NMS

At every training step, the Hungarian algorithm finds the **one-to-one** assignment between the N queries
and the M objects (M ≤ N) that minimises a cost of class probability, L1 box distance and GIoU. Matched
queries are trained toward their object and all others toward "no object". Each object therefore has
exactly one positive query, and a second query on the same object is penalised as background.
Duplicates are suppressed by the loss, so inference can rank the N query outputs directly with **no NMS**.
This also avoids NMS removing true neighbours in crowds.

**Training dynamics.** One positive per object (vs ~10 in TAL, or dozens of anchors) gives a sparse,
noisy signal, and early on the matching itself is unstable: a query can switch objects between steps.
DETR-style models therefore converge much more slowly than CNN detectors. RT-DETR mitigates this with
auxiliary losses on every decoder layer, query selection from encoder features (IoU-aware) and
denoising queries (noised ground-truth boxes that bypass matching). The cap N on queries bounds how many
objects a frame can contain.

In [ ]:
# Setup: code + extra packages. On Kaggle the repo comes from an attached dataset or GitHub.
import os, sys, shutil, subprocess
from pathlib import Path

GITHUB_REPO = "ivan-in-a-nutshell/AI_HW_2"   # your repository; if private, add a GITHUB_TOKEN Kaggle secret
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    REPO = Path("/kaggle/working/repo")
    if not REPO.exists():
        code = [p.parent for p in Path("/kaggle/input").glob("*/pyproject.toml")]
        if code:                                   # repo uploaded as a Kaggle dataset
            shutil.copytree(code[0], REPO)
        else:                                      # clone from GitHub
            try:
                from kaggle_secrets import UserSecretsClient
                auth = UserSecretsClient().get_secret("GITHUB_TOKEN") + "@"
            except Exception:
                auth = ""
            subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{GITHUB_REPO}.git", str(REPO)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-r", str(REPO / "requirements.txt")], check=True)
    sys.path.insert(0, str(REPO / "src"))
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
print("repo:", REPO)

In [ ]:
# Data and the fixed benchmark protocol: fold 0, every 10th frame, unmasked test-scale validation frames
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from visdet.data import find_raw_root, find_prepared, read_prepared, fold_split
from visdet.evaluate import build_gt
from visdet.plotting import setup_style, save, SERIES
from visdet.results import load_log, family_table, best_run, plot_histories, plot_breakdown, styled

RAW, PREP = find_raw_root(), find_prepared()
RUNS = Path("/kaggle/working/runs") if ON_KAGGLE else REPO / "runs"
FOLD, STEP = 0, 10
images, boxes, folds = read_prepared(PREP)
train_rows, val_rows = fold_split(images, folds, FOLD, train_step=STEP, val_step=STEP)
gt = build_gt(val_rows, boxes)
N_GPU = torch.cuda.device_count()
SMOKE = os.environ.get("VISDET_SMOKE") == "1"   # CPU dry run: 1 epoch on a few frames, nothing logged for real
if SMOKE:
    val_rows = val_rows.head(8); gt = build_gt(val_rows, boxes)
setup_style()
print(f"fold {FOLD}: {len(train_rows)} training frames, {len(val_rows)} validation frames; {N_GPU} GPU(s)")

In [ ]:
# Ultralytics dataset files for fold 0
from visdet.pipeline import find_weights
from visdet.ul_utils import write_yolo_data, train_ultralytics, evaluate_ultralytics
DATA_YAML = write_yolo_data(PREP, images, folds, FOLD, RUNS / "data_fold0", train_step=STEP, val_step=STEP,
                            val_limit=16 if SMOKE else None)
DEVICE = ",".join(str(i) for i in range(N_GPU)) or "cpu"
TRAIN_KW = {"device": DEVICE, "workers": 4, "verbose": False}
UL_SMOKE = {"imgsz": 320, "epochs": 1, "time": None, "batch": 4, "fraction": 0.005, "workers": 0} if SMOKE else {}

In [ ]:
# Train each configuration on the same 2 GPU-hour budget as YOLO11s, then evaluate
CONFIGS = ['rtdetr_base', 'rtdetr_q500', 'rtdetr_matcher', 'rtdetr_bblr']
for name in CONFIGS:
    try:                                   # this session or an attached earlier output of this notebook
        weights = find_weights(name, RUNS); print(f"{name}: reusing {weights}")
    except FileNotFoundError:
        weights, cfg = train_ultralytics(f"configs/{name}.yaml", DATA_YAML, RUNS, **(TRAIN_KW | UL_SMOKE))
        evaluate_ultralytics(weights, cfg, RAW, val_rows, boxes, fold=FOLD, **({"imgsz": 320} if SMOKE else {}))

## Results against the CNN detector with the same budget

In [ ]:
# RT-DETR runs + YOLO11s baseline (same budget) + tuned YOLO
log = load_log()
yolo_runs = [r for r in ['yolo11s_base', 'yolo11s_960', 'yolo11s_1536', 'yolo11s_p2', 'yolo11s_tal'] if r in log.index]
compare = CONFIGS + (["yolo11s_base", best_run(log, yolo_runs)] if yolo_runs else [])
styled(family_table(log, list(dict.fromkeys(compare))))

In [ ]:
# Validation mAP over GPU time: RT-DETR vs YOLO11s under the same 2 GPU-hour budget
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.4))
plot_histories([r for r in ["yolo11s_base", "rtdetr_base", best_run(log, CONFIGS)] if r in log.index],
               x="gpu_hours", ax=a, title="Validation mAP vs GPU time (same budget)")
dens = [c for c in log.columns if "dens" in c]
plot_breakdown(log, [r for r in ["yolo11s_base", "rtdetr_base", best_run(log, CONFIGS)] if r in log.index],
               cols=["AP_s", "AP_m", "AP_l"] + dens, labels=["small", "medium", "large"] + [c.replace("AP_dens", "") + " obj" for c in dens],
               ax=b, title="AP by size and objects per frame")
fig.tight_layout(); save(fig, "rtdetr_vs_yolo")

## Failure analysis

Confident detections (score ≥ 0.3) of the best run are labelled with an error type and every object
as found / missed (`visdet.analysis`): background false positives, wrong class, poor localisation,
duplicates, and misses broken down by size and occlusion.

In [ ]:
# Predict the validation frames with the best run and label every error
from visdet.analysis import classify_errors, error_summary, confusion, frame_errors, draw_errors
from visdet.infer import predict_eval_frames, load_frame
from visdet.infer import UltralyticsPredictor
BEST = best_run(log, CONFIGS)
predictor = UltralyticsPredictor(find_weights(BEST, RUNS), imgsz=1280, max_det=500, rtdetr=True)
dets = predict_eval_frames(predictor, RAW, val_rows, desc="validation")
det_t, gt_t = classify_errors(boxes[boxes.image_id.isin(val_rows.image_id)], dets, score_thr=0.3)
shares, recall = error_summary(det_t, gt_t)
display(shares.to_frame("share of confident detections").T.round(3))
display(recall.round(3))
confusion(det_t)

In [ ]:
# Three failure examples: most missed objects, most class confusions, most background false positives
fe = frame_errors(det_t, gt_t)
picks = {"missed objects": fe.missed.idxmax(),
          "class confusion": fe.get("wrong_cls", fe.missed).idxmax(),
          "background false positives": fe.get("background", fe.missed).idxmax()}
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, (what, img_id) in zip(axes, picks.items()):
    r = val_rows.set_index("image_id").loc[img_id]
    frame = load_frame(RAW / r.file_name, (r.out_w, r.out_h))
    vis = draw_errors(frame, gt_t[gt_t.image_id == img_id], det_t[det_t.image_id == img_id])
    ax.imshow(vis[:, :, ::-1]); ax.axis("off")
    ax.set_title(f"{what}: {r['clip']} #{r.frame} ({fe.loc[img_id, 'missed']}/{fe.loc[img_id, 'objects']} missed)", fontsize=8)
fig.suptitle("green = found, yellow = missed, red = background FP, orange = wrong class, magenta = poor localisation", fontsize=8)
fig.tight_layout()
save(fig, "rtdetr_failures", photo=True)

## Findings

_Fill in after the runs:_ whether RT-DETR converged within the budget (is the curve still rising at
the end?), small-object and dense-scene AP against YOLO11s, the effect of 500 queries (AR, dense
clips), of the matching cost and of the backbone learning rate, and the error types (duplicates should
be rare without NMS; misses in crowds if the queries run out).

**After the session:** download `/kaggle/working/repo/experiments/` (metrics, per-epoch histories) and
`images/` from the notebook output and commit them, so later notebooks and the report can use them.
Checkpoints stay in the notebook output (`/kaggle/working/runs`).